## Traitement de données et extraction de caractéristiques

In [2]:
import librosa
import numpy as np
import scipy.signal

In [3]:
def bandpass_filter(data, lowcut=100.0, highcut=2000.0, fs=22050, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = scipy.signal.butter(order, [low, high], btype='band')
    
    y = scipy.signal.lfilter(b, a, data)
    return y

In [4]:
def extract_features(file_path, target_sr=22050, fixed_duration_s=5.0):
    # Chargement et standardisation de l'audio
    y, sr = librosa.load(file_path, sr=target_sr)
    
    # Suppression des silences aux extrémités
    y_trimmed, _ = librosa.effects.trim(y, top_db=20)
    
    # Filtrage des bruits de fond
    y_filtered = bandpass_filter(y_trimmed, fs=sr)
    
    # Standardisation de la durée (Padding ou Cropping)
    target_length = int(target_sr * fixed_duration_s)
    if len(y_filtered) < target_length:
        y_final = np.pad(y_filtered, (0, target_length - len(y_filtered)), mode='constant')
    else:
        y_final = y_filtered[:target_length]
        
    # Spectrogramme Mel et conversion en dB
    mel_spec = librosa.feature.melspectrogram(y=y_final, sr=sr, n_mels=128, n_fft=2048, hop_length=512)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    # Features supplémentaires : MFCC, Spectral Centroid, Zero Crossing Rate
    mfcc = librosa.feature.mfcc(y=y_final, sr=sr, n_mfcc=13)
    spectral_centroid = librosa.feature.spectral_centroid(y=y_final, sr=sr)
    zcr = librosa.feature.zero_crossing_rate(y_final)
    
    return mel_spec_db, mfcc, spectral_centroid, zcr

## Visualisation de données traitées

In [5]:
extracted_features = extract_features("data/Asthma Detection Dataset Version 2/Asthma Detection Dataset Version 2/asthma/P1AsthmaIE_1.wav")

c:\Users\Coaraa\Documents\Cours\UQAC\Hiver 2026\AtelierPratique\hackathon\respiratory-disease-detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
print("Mel Spectrogram (dB) shape:", extracted_features[0].shape)
print("MFCC shape:", extracted_features[1].shape)
print("Spectral Centroid shape:", extracted_features[2].shape)
print("Zero Crossing Rate shape:", extracted_features[3].shape)

Mel Spectrogram (dB) shape: (128, 216)
MFCC shape: (13, 216)
Spectral Centroid shape: (1, 216)
Zero Crossing Rate shape: (1, 216)


In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

def visualize_audio_changes(y_original, sr_original, y_processed, sr_processed):
    """
    Affiche la forme d'onde et le spectrogramme Mel avant et après prétraitement.
    """
    fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(14, 10))
    fig.suptitle("Comparaison de l'Audio : Original vs Prétraité", fontsize=16)

    # --- 1. Forme d'onde AVANT ---
    librosa.display.waveshow(y_original, sr=sr_original, ax=ax[0, 0])
    ax[0, 0].set(title='Forme d\'onde Originale (avec silences)')
    ax[0, 0].label_outer()

    # --- 2. Forme d'onde APRÈS ---
    librosa.display.waveshow(y_processed, sr=sr_processed, ax=ax[0, 1], color='g')
    ax[0, 1].set(title='Forme d\'onde Prétraitée (silences coupés, durée fixe)')
    ax[0, 1].label_outer()

    # --- 3. Spectrogramme AVANT ---
    S_orig = librosa.feature.melspectrogram(y=y_original, sr=sr_original, n_mels=128)
    S_dB_orig = librosa.power_to_db(S_orig, ref=np.max) # Conversion en dB recommandée
    img1 = librosa.display.specshow(S_dB_orig, x_axis='time', y_axis='mel', sr=sr_original, ax=ax[1, 0])
    ax[1, 0].set(title='Spectrogramme Original (avec bruit de fond)')
    fig.colorbar(img1, ax=ax[1, 0], format='%+2.0f dB')

    # --- 4. Spectrogramme APRÈS (Filtre passe-bande 100-2000Hz appliqué) ---
    S_proc = librosa.feature.melspectrogram(y=y_processed, sr=sr_processed, n_mels=128)
    S_dB_proc = librosa.power_to_db(S_proc, ref=np.max)
    img2 = librosa.display.specshow(S_dB_proc, x_axis='time', y_axis='mel', sr=sr_processed, ax=ax[1, 1])
    ax[1, 1].set(title='Spectrogramme Prétraité (Fréquences isolées)')
    fig.colorbar(img2, ax=ax[1, 1], format='%+2.0f dB')

    plt.tight_layout()
    plt.show()